<a href="https://colab.research.google.com/github/nahomaron/ai-surveillance/blob/m0-implementation/research/colab/s2_detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RQ1 — S2 detection on Colab GPU (Owner decision D-60)

Runs the **frozen D-45 detector** (YOLO-World v2-s, `yolov8s-worldv2.pt`, prompts `person` and `forklift`, every box with score ≥ 0.05 stored; the thresholds person 0.20 / forklift 0.05 are applied later on the laptop) on the videos of one job manifest, and writes one raw file per camera to Google Drive in the `out/m1b/pilot/s2raw/` format.

- **Jobs:** `equivalence` (2 synthetic tuning cameras: run this first) and `tier1` (the 45 held-out-layout cameras, only after the laptop has reported that the equivalence check passed).
- **Checks before use:** the SHA-256 of both weight files and of every input video against the manifest. Any difference stops the run.
- **Resumable:** after a disconnect, run every cell again from the top. Finished cameras are skipped.
- **D-60 limits:** nothing downstream of S2 runs here. For `tier1`, do not open, plot or count the output files: the laptop does the integrity checks. This notebook prints frame counts and file hashes only.

Instructions: `research/colab/HOWTO.md`.

## 1. Settings (change `JOB` only)

In [1]:
JOB = "equivalence"          # "equivalence" first; "tier1" only after the laptop reports a pass
REPO_REF = "m0-implementation"  # branch or commit of github.com/nahomaron/ai-surveillance
DRIVE_ROOT = "/content/drive/MyDrive/rq1_colab"
assert JOB in ("equivalence", "tier1")

## 2. Mount Google Drive and check the GPU

In [2]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
assert os.path.isdir(f"{DRIVE_ROOT}/data"), f"upload the videos to {DRIVE_ROOT}/data first (HOWTO step 1)"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), "choose Runtime > Change runtime type > GPU, then run again from the top"
print("torch", torch.__version__, "CUDA", torch.version.cuda, "device", torch.cuda.get_device_name(0))

ValueError: mount failed

## 3. Install the pinned packages and the runner (torch and CUDA stay as Colab provides them)

In [ ]:
!pip install -q "ultralytics==8.4.215" "av==19.0.2" "git+https://github.com/ultralytics/CLIP.git@7ffa84b3bfa40c42ecc1c77147a855e69cb2dd40"
!rm -rf /content/repo && git clone -q --depth 1 --branch {REPO_REF} https://github.com/nahomaron/ai-surveillance.git /content/repo
!pip install -q --no-deps /content/repo/research/colab
COMMIT = subprocess.run(["git", "-C", "/content/repo", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
os.environ["RQ1_RUNNER_COMMIT"] = COMMIT
os.environ["YOLO_CONFIG_DIR"] = "/content/ultralytics_config"
os.environ["YOLO_AUTOINSTALL"] = "false"
import ultralytics, av
assert ultralytics.__version__ == "8.4.215" and av.__version__ == "19.0.2", (ultralytics.__version__, av.__version__)
OUT = f"{DRIVE_ROOT}/out/{JOB}"
os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/pip_freeze_{COMMIT[:12]}.txt", "w") as f:
    f.write(subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout)
print("runner commit", COMMIT)

## 4. Check every input video and the weights (no detection yet)

In [ ]:
JOB_MANIFEST = f"/content/repo/research/colab/{JOB}_manifest.json"
WEIGHTS = f"{DRIVE_ROOT}/weights"   # downloaded once from the published URLs, then checked by SHA-256
!python -m rq1_colab.runner --job {JOB_MANIFEST} --data-root {DRIVE_ROOT}/data --out {OUT} --weights-dir {WEIGHTS} --device 0 --verify-only

## 5. Run the detector (re-run this cell after a disconnect: finished cameras are skipped)

In [ ]:
!python -m rq1_colab.runner --job {JOB_MANIFEST} --data-root {DRIVE_ROOT}/data --out {OUT} --weights-dir {WEIGHTS} --device 0

## 6. Status (counts of cameras and frames only)

In [ ]:
import json
m = json.load(open(f"{OUT}/colab_manifest.json"))
print(JOB, "complete" if m["complete"] else "INCOMPLETE: run step 5 again", "-", m["cameras_finished"], "of", m["cameras_in_job"], "cameras,",
      m["frames_total"], "frames,", m["wall_s_total"], "s on", m["environment"]["device_name"])
print("Next: download the folder", OUT, "and put it at out/colab/" + JOB + "/ on the laptop (HOWTO step 3 or 5).")